# Ingest circuits.csv file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
circuits_df = (
    spark.read
         .format('csv')
         .option('header', 'true')
#         .option('inferSchema', 'true')
         .option('mode', 'FAILFAST')
         .schema(circuits_schema)
         .load(source_file)
)

In [0]:
# display(circuits_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
# display(circuits_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
# display(spark.table(table_name))

In [0]:
# display(spark.table('formula1.bronze.circuits'))
# commenting display method so that jobs can complete faster

In [0]:
%sql
--select count(1) as count from formula1.bronze.circuits

count
82


In [0]:
%sql
--describe table formula1.bronze.circuits

col_name,data_type,comment
circuitId,string,null
url,string,null
circuitName,string,null
lat,double,null
long,double,null
locality,string,null
country,string,null
ingestion_timestamp,timestamp,null
source_file,string,null
# Clustering Information,,


In [0]:
%sql
--DESCRIBE HISTORY formula1.bronze.circuits

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-07T13:37:56.000Z,143528513934144,basuhiregoudra04@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""097fb399-9321-44e1-8711-89e519370c0a"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e00e564f-bf1f-482c-a25c-fcb8fee62471"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-72072c30-1c9b-4264-ae70-044f76b00aee""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1781364196734429),91d63476-f9d9-47f8-bba7-5dbb8df119b9,1107-130144-sm80dfxf-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 82, numOutputBytes -> 7996)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
